# Frozen HetNet policy under UE mobility

Load the final small/flat checkpoint and evaluate **50 movement updates without training**. Iteration 0 is the static baseline; iterations 1–50 each move users, let the policy select a new allocation using the last observed state, then measure QoS. This enriched policy observes global QoS, relative BS distances, candidate received powers, and previous association/channel/SINR feedback. This tests allocation quality, not global optimality.

Choose the `~/venvs/PyTorch` Python kernel and run all cells. Training/simulator source files are unchanged. Each execution creates a separate result directory. The last cells provide a standalone slider/player, a GIF, individual PNG frames and CSV metrics.


In [ ]:
from pathlib import Path
import sys, json, random, hashlib, csv, base64, io
from datetime import datetime
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from PIL import Image
from IPython.display import display, HTML

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rl/checkpoints.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the UARA-DRL repository")
sys.path.insert(0, str(ROOT))
from rl.interaction import greedy_actions as select_greedy, move_users
from rl.evaluation import restore_policy
from rl.training import compute_rewards_and_next_state
from telecom import mobility

CHECKPOINT = ROOT / "Result/small_enriched_mobility_30ue_large_cells_seed42_20261010-153106/checkpoints/final.pt"
MOBILITY_SEED = 123
ITERATIONS = 50
UE_SPEED_METRES_PER_UPDATE = 15.0
DECISIONS_PER_MOVEMENT = 5
BASELINE_MAX_DECISIONS = 100
FRAME_DURATION_MS = 450
DEVICE = "cpu"
assert ITERATIONS > 0 and DECISIONS_PER_MOVEMENT > 0 and UE_SPEED_METRES_PER_UPDATE > 0
print("Checkpoint:", CHECKPOINT)


## Restore and freeze the trained policy
The existing loader restores agents, base stations, architecture and positions. Inference uses greedy argmax actions, disabled gradients and disabled dropout. Parameter hashes and replay sizes are checked after evaluation.


In [ ]:
random.seed(MOBILITY_SEED)
np.random.seed(MOBILITY_SEED)
torch.manual_seed(MOBILITY_SEED)
agents, scenario, encoder, opt, sce, checkpoint = restore_policy(CHECKPOINT, DEVICE)
models = [m for a in agents for m in (a.model_policy, a.model_target)]
if encoder.enabled:
    models.append(encoder._encoder)

def weights_hash():
    h = hashlib.sha256()
    for model in models:
        for name, tensor in model.state_dict().items():
            h.update(name.encode())
            h.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

original_hash = weights_hash()
original_replay_sizes = [len(a.memory) for a in agents]
assert all(not m.training for m in models)
assert all(not p.requires_grad for m in models for p in m.parameters())
initial_positions = np.array([a.location.cpu().numpy().copy() for a in agents])
bs_list = scenario.Get_BaseStations()
bs_positions = np.array([b.Get_Location() for b in bs_list])
print(f"Restored {len(agents)} agents; training seed {opt.seed}; completed episodes {checkpoint['episode'] + 1}")


## Evaluate the static baseline and movement sequence
The static baseline starts from zero QoS, as training episodes do, and allows up to 100 allocation decisions to reach full QoS. It is excluded from mobility summary statistics.

For each movement update, the policy selects an allocation using the last QoS observation, then QoS is measured at the new positions. There is no measurement between movement and the first allocation decision. If multiple decisions are enabled, subsequent decisions use feedback from the preceding decision. The saved QoS is measured **after** the final decision. All 50 updates execute, regardless of QoS. The existing random-waypoint/hotspot model is reused; its “episode” update is treated here as a movement update, with no assumed physical time duration.

Only base-station geometry is cached by the radio calculator; UE-dependent received power and interference are recomputed on every measurement. Notebook-local mobility settings are restored in `finally`.


In [ ]:
def greedy_actions(state, previous, capacity=None):
    return select_greedy(agents, scenario, state, encoder, previous, capacity)

records, snapshots, baseline_trace = [], [], []
def record(iteration, actions, rewards, qos, capacity, old_actions, old_positions):
    positions = np.array([a.location.cpu().numpy().copy() for a in agents])
    selected = actions.cpu().numpy().copy()
    row = dict(iteration=iteration, qos_fraction=qos.mean().item(),
               satisfied_users=int(qos.sum().item()), mean_reward=rewards.mean().item(),
               mean_capacity_mbps=capacity.mean().item(),
               association_changes=int(np.sum(selected // sce.nChannel != old_actions // sce.nChannel)),
               channel_changes=int(np.sum(selected % sce.nChannel != old_actions % sce.nChannel)),
               mean_movement_m=float(np.linalg.norm(positions - old_positions, axis=1).mean()))
    records.append(row)
    snapshots.append(dict(positions=positions, actions=selected, qos=qos.cpu().numpy().copy(),
                          capacity=capacity.cpu().numpy().copy()))

with torch.inference_mode():
    state = torch.zeros(len(agents), device=DEVICE)
    actions = None
    capacity = None
    for decision in range(BASELINE_MAX_DECISIONS):
        actions = greedy_actions(state, actions, capacity)
        rewards, qos, state, capacity, _ = compute_rewards_and_next_state(agents, actions, state, scenario)
        baseline_trace.append(dict(decision=decision+1, qos_fraction=qos.mean().item()))
        if bool(qos.eq(1).all()):
            break
    record(0, actions, rewards, qos, capacity, actions.cpu().numpy(), initial_positions)

    mover = mobility.MobilityManager(scenario, len(agents), seed=MOBILITY_SEED,
                                    enabled=True, speed=UE_SPEED_METRES_PER_UPDATE)
    for i, position in enumerate(initial_positions):
        mover.set_ue_position(i, position)
    for iteration in range(1, ITERATIONS + 1):
        old_positions = np.array([a.location.cpu().numpy().copy() for a in agents])
        old_actions = actions.cpu().numpy().copy()
        move_users(agents, mover)
        for _ in range(DECISIONS_PER_MOVEMENT):
            actions = greedy_actions(state, actions, capacity)
            rewards, qos, state, capacity, _ = compute_rewards_and_next_state(agents, actions, state, scenario)
        record(iteration, actions, rewards, qos, capacity, old_actions, old_positions)

assert weights_hash() == original_hash, "Model weights changed"
assert [len(a.memory) for a in agents] == original_replay_sizes, "Replay memory changed"
assert len(records) == ITERATIONS + 1
assert any(r['mean_movement_m'] > 0 for r in records[1:]), "UEs did not move"
assert all(np.isfinite(r['qos_fraction']) for r in records)
assert all(0 <= r['qos_fraction'] <= 1 for r in records)
assert all(np.linalg.norm(b['positions'] - a['positions'], axis=1).max()
           <= UE_SPEED_METRES_PER_UPDATE + 0.001 for a, b in zip(snapshots, snapshots[1:]))
print(f"Static baseline: {records[0]['qos_fraction']:.1%} QoS after {len(baseline_trace)} decisions")
print(f"Mobility QoS: mean={np.mean([r['qos_fraction'] for r in records[1:]]):.1%}, "
      f"minimum={min(r['qos_fraction'] for r in records[1:]):.1%}, final={records[-1]['qos_fraction']:.1%}")
print("Verified: weights unchanged, replay unchanged, 50 updates completed" if ITERATIONS == 50 else "Verified: weights/replay unchanged; all updates completed")


## Save metrics and full per-user trajectories
`metrics.csv` contains iteration-level results; `users.csv` contains positions, chosen BS/channel, QoS and capacity for every UE. Indices start at zero. Metadata records the source checkpoint hash and movement settings. These are inference metrics, separate from exploratory training results.


In [ ]:
OUTPUT = ROOT / "Result" / (f"mobility_small_{getattr(opt, 'observation_mode', 'qos')}_{len(agents)}ue_seed{opt.seed}_" + datetime.now().strftime("%Y%m%d-%H%M%S-%f"))
OUTPUT.mkdir(parents=True, exist_ok=False)
(OUTPUT / "frames").mkdir()
def write_csv(path, rows):
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
write_csv(OUTPUT / "metrics.csv", records)
write_csv(OUTPUT / "baseline.csv", baseline_trace)
users = [dict(iteration=t, ue=i, x=float(s['positions'][i,0]), y=float(s['positions'][i,1]),
              base_station=int(s['actions'][i] // sce.nChannel), channel=int(s['actions'][i] % sce.nChannel),
              qos=int(s['qos'][i]), capacity_mbps=float(s['capacity'][i]))
         for t, s in enumerate(snapshots) for i in range(len(agents))]
write_csv(OUTPUT / "users.csv", users)
metadata = dict(checkpoint=str(CHECKPOINT), checkpoint_sha256=hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest(),
                training_seed=opt.seed, mobility_seed=MOBILITY_SEED, iterations=ITERATIONS,
                ue_speed_metres_per_update=UE_SPEED_METRES_PER_UPDATE, decisions_per_movement=DECISIONS_PER_MOVEMENT,
                baseline_decisions=len(baseline_trace), early_termination=False, learning=False,
                policy="greedy", fresh_qos_after_movement=False, observation_timing="last post-allocation QoS", weights_unchanged=True,
                replay_unchanged=True, initial_frame_is_static_baseline=True,
                mobility_constants={k: getattr(mobility,k) for k in ["HOTSPOT_ENABLED", "HOTSPOT_ATTRACTION_PROBABILITY",
                    "HOTSPOT_COUNT_RANGE", "HOTSPOT_RADIUS_RANGE", "HOTSPOT_WEIGHT_RANGE", "HOTSPOT_SPEED",
                    "HOTSPOT_LIFETIME_RANGE", "UE_PAUSE_EPISODES", "CONSTRAIN_TO_COVERAGE"]})
(OUTPUT / "metadata.json").write_text(json.dumps(metadata, indent=2))
print("Results:", OUTPUT)


## Network frames, QoS history and animation
Base-station marker shape identifies its tier; dashed circles show coverage. UE colour indicates QoS satisfaction. Lines show the selected serving BS and are coloured by channel; UE labels give `UE index : channel`. Grey trails show recent movement. Iteration 0 is the settled static baseline. Connections can be selected outside coverage—these are shown rather than silently corrected.


In [ ]:
plt.rcParams.update({"font.size": 10})
bounds = (min(b.Get_Location()[0]-b.radius for b in bs_list), max(b.Get_Location()[0]+b.radius for b in bs_list),
          min(b.Get_Location()[1]-b.radius for b in bs_list), max(b.Get_Location()[1]+b.radius for b in bs_list))
channel_map = plt.get_cmap("hsv", sce.nChannel)
frame_paths = []
for t, snap in enumerate(snapshots):
    fig, (ax, curve) = plt.subplots(1, 2, figsize=(15, 7), gridspec_kw={"width_ratios": [1.55, 1]}, constrained_layout=True)
    for b, xy in zip(bs_list, bs_positions):
        marker = {"MBS": "^", "PBS": "s", "FBS": "D"}[b.bs_type]
        ax.add_patch(Circle(xy, b.radius, fill=False, ls="--", lw=.65, alpha=.22, color="slategray"))
        ax.scatter(*xy, marker=marker, s=80, c="navy", zorder=4)
        ax.annotate(f"{b.bs_type}{b.id}", xy, xytext=(4,5), textcoords="offset points", fontsize=7)
    for i, xy in enumerate(snap['positions']):
        bs_id, channel = divmod(int(snap['actions'][i]), sce.nChannel)
        dest = bs_positions[bs_id]
        ax.plot([xy[0],dest[0]], [xy[1],dest[1]], color=channel_map(channel), alpha=.5, lw=.9)
        trail = np.array([s['positions'][i] for s in snapshots[max(0,t-5):t+1]])
        ax.plot(trail[:,0], trail[:,1], color="gray", lw=.7, alpha=.6)
        ax.scatter(*xy, c="seagreen" if snap['qos'][i] else "crimson", s=28, edgecolors="white", linewidths=.3, zorder=5)
        ax.annotate(f"{i}:{channel}", xy, xytext=(3,-8), textcoords="offset points", fontsize=6)
    ax.set(xlim=(bounds[0]-30,bounds[1]+30), ylim=(bounds[2]-30,bounds[3]+30), xlabel="x (m)", ylabel="y (m)", aspect="equal")
    handles=[Line2D([],[], marker=m, color="navy", ls="", label=n) for m,n in [('^','MBS'),('s','PBS'),('D','FBS')]]
    handles += [Line2D([],[], marker='o', color=c, ls='', label=n) for c,n in [('seagreen','QoS met'),('crimson','QoS unmet')]]
    ax.legend(handles=handles, loc="upper right", fontsize=8)
    ax.set_title("Selected connections | UE labels: index:channel")
    curve.plot(range(t+1), [100*r['qos_fraction'] for r in records[:t+1]], color="teal", marker=".", label="After policy decision")
    curve.axhline(100, color="gray", ls="--", lw=.7)
    curve.set(xlim=(0,ITERATIONS), ylim=(0,105), xlabel="Mobility iteration (0 = static baseline)", ylabel="UEs satisfying QoS (%)")
    curve.grid(alpha=.2); curve.legend(loc="lower left", fontsize=8)
    curve.set_title(f"QoS {records[t]['satisfied_users']}/{len(agents)} | mean reward {records[t]['mean_reward']:.3f}")
    fig.suptitle(f"Frozen {getattr(opt, 'observation_mode', 'qos')} policy · training seed {opt.seed} · mobility seed {MOBILITY_SEED} · iteration {t}/{ITERATIONS}")
    path = OUTPUT / "frames" / f"frame_{t:03d}.png"
    fig.savefig(path, dpi=100)
    plt.close(fig)
    frame_paths.append(path)
images = [Image.open(p).convert("RGB") for p in frame_paths]
images[0].save(OUTPUT / "mobility.gif", save_all=True, append_images=images[1:], duration=FRAME_DURATION_MS, loop=0)
for img in images: img.close()
fig, ax = plt.subplots(figsize=(10,4), constrained_layout=True)
ax.plot([r['iteration'] for r in records], [r['qos_fraction']*100 for r in records], marker='.', color='teal')
ax.set(xlabel="Mobility iteration (0 = static baseline)", ylabel="QoS satisfaction (%)", ylim=(0,105), title="Frozen policy under user mobility")
ax.grid(alpha=.2)
fig.savefig(OUTPUT / "qos_history.png", dpi=150)
plt.close(fig)
print(f"Saved {len(frame_paths)} frames, mobility.gif and qos_history.png")


## Watch the sequence
The self-contained HTML player works without a Jupyter widget extension. Use Play/Pause or move its slider. `mobility.gif` can also be opened directly. This is a single movement trajectory; it does not establish general performance across mobility seeds.


In [ ]:
encoded = ["data:image/png;base64," + base64.b64encode(p.read_bytes()).decode() for p in frame_paths]
player = """<!doctype html><html><meta charset="utf-8"><title>HetNet mobility evaluation</title>
<style>body{font-family:system-ui;margin:20px;background:#f5f7fa}img{width:100%;max-width:1500px}input{width:65%}button{padding:8px 18px}</style>
<h2>Frozen HetNet policy: network and QoS</h2>
<button id="play">Play</button> <input id="slider" type="range" min="0" max="__MAX__" value="0"> <span id="label"></span><br>
<img id="frame" alt="Network topology and QoS history">
<script>const frames=__FRAMES__;const slider=document.getElementById('slider');let timer=null;
function show(){document.getElementById('frame').src=frames[Number(slider.value)];document.getElementById('label').textContent='Iteration '+slider.value+' / '+slider.max;}
slider.oninput=show;document.getElementById('play').onclick=function(){if(timer){clearInterval(timer);timer=null;this.textContent='Play';}else{this.textContent='Pause';timer=setInterval(()=>{slider.value=(Number(slider.value)+1)%frames.length;show();},__DELAY__);}};show();</script></html>"""
player = player.replace('__MAX__', str(ITERATIONS)).replace('__FRAMES__', json.dumps(encoded)).replace('__DELAY__', str(FRAME_DURATION_MS))
(OUTPUT / "player.html").write_text(player)
import html
# srcdoc keeps the player usable inside a trusted notebook without external paths.
display(HTML('<iframe style="width:100%;height:850px;border:0" srcdoc="' + html.escape(player, quote=True) + '"></iframe>'))
print("Standalone player:", OUTPUT / "player.html")
